In [41]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [42]:
from datasets import load_dataset, DatasetDict
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd
import json
import re
import hashlib

In [43]:
DATASET_ID = "lighteval/okapi_mmlu"
CONFIG_NAME = "vi"

TASK_NAME = "wiki_mcq"
SOURCE_NAME = "lighteval/okapi_mmlu:vi"
DIFFICULTY = "medium"

OUT_DIR = Path("seed_exports/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL = OUT_DIR / "wiki_mcq_seed.jsonl"
OUT_REJECTS_JSONL = OUT_DIR / "wiki_mcq_seed_rejects.jsonl"
OUT_REPORT_JSON = OUT_DIR / "wiki_mcq_seed_report.json"

In [44]:
# load dataasets 
ds = load_dataset(DATASET_ID, CONFIG_NAME)
ds

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'subject', 'id'],
        num_rows: 13062
    })
    validation: Dataset({
        features: ['question', 'choices', 'answer', 'subject', 'id'],
        num_rows: 1456
    })
    dev: Dataset({
        features: ['question', 'choices', 'answer', 'subject', 'id'],
        num_rows: 271
    })
})

In [45]:
ds.items()

dict_items([('test', Dataset({
    features: ['question', 'choices', 'answer', 'subject', 'id'],
    num_rows: 13062
})), ('validation', Dataset({
    features: ['question', 'choices', 'answer', 'subject', 'id'],
    num_rows: 1456
})), ('dev', Dataset({
    features: ['question', 'choices', 'answer', 'subject', 'id'],
    num_rows: 271
}))])

In [46]:
for split_name, split_ds in ds.items():
    print(f"\n{split_name}")
    print("num_rows:", len(split_ds))
    print("columns:", split_ds.column_names)
    print("features:", split_ds.features)


test
num_rows: 13062
columns: ['question', 'choices', 'answer', 'subject', 'id']
features: {'question': Value('string'), 'choices': List(Value('string')), 'answer': Value('string'), 'subject': Value('string'), 'id': Value('string')}

validation
num_rows: 1456
columns: ['question', 'choices', 'answer', 'subject', 'id']
features: {'question': Value('string'), 'choices': List(Value('string')), 'answer': Value('string'), 'subject': Value('string'), 'id': Value('string')}

dev
num_rows: 271
columns: ['question', 'choices', 'answer', 'subject', 'id']
features: {'question': Value('string'), 'choices': List(Value('string')), 'answer': Value('string'), 'subject': Value('string'), 'id': Value('string')}


In [47]:
# Xem vài sample đầu tiên
for split_name, split_ds in ds.items():
    print(f"\nSAMPLE FROM {split_name}")
    sample = split_ds[1]
    for k, v in sample.items():
        print(f"{k}: {v}")
    break


SAMPLE FROM test
question: Nhóm Quốc gia Islam kêu gọi đến:
choices: ['Những người nhập cư thế hệ thứ hai sinh ra tại Anh từ lục địa châu Á', 'Những người da trắng người Mỹ muốn chuyển đổi sang Hồi giáo', "Những người Mỹ gốc Phi cảm thấy bị loại trừ khỏi 'nồi hỗn hợp sắc tộc' ở Mỹ", 'Các người Caribê thuộc dân tộc Châu Phi sống trong các khu vực nội thành và có một văn hóa trẻ đặc trưng']
answer: C
subject: sociology
id: sociology/test/49


In [48]:
# dataset dict to df
frames = []
for split_name, split_ds in ds.items():
    df_split = split_ds.to_pandas()
    df_split['original_split'] = split_name
    frames.append(df_split)
    
df_raw = pd.concat(frames, ignore_index=True)

In [49]:
df_raw.head()

,question,choices,answer,subject,id,original_split
0,"Trong hai tình huống sau đây, nhân vật chính (...","[Sai, Sai, Sai, Không sai, Không sai, Sai, Khô...",B,moral_scenarios,moral_scenarios/test/106,test
1,Nhóm Quốc gia Islam kêu gọi đến:,[Những người nhập cư thế hệ thứ hai sinh ra tạ...,C,sociology,sociology/test/49,test
2,Phần nào của phổ điện từ có bước sóng ngắn nhất?,"[Tia gamma, Tia X, Sóng vô tuyến, Sóng viễn th...",A,miscellaneous,miscellaneous/test/535,test
3,Mục đích chính của sổ chi tiết hoạt động của m...,"[Nhà cung cấp tài nguyên., Quản lý., Người hưở...",A,professional_accounting,professional_accounting/test/251,test
4,Các ông bà có cháu đang theo học đại học cho b...,[Hỏi về sự hài lòng với cuộc sống của họ và đề...,C,human_aging,human_aging/test/173,test


In [50]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 14789 entries, 0 to 14788
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   question        14789 non-null  str   
 1   choices         14789 non-null  object
 2   answer          14789 non-null  str   
 3   subject         14789 non-null  str   
 4   id              14789 non-null  str   
 5   original_split  14789 non-null  str   
dtypes: object(1), str(5)
memory usage: 5.8+ MB


In [51]:
print("Rows by split:")
display(df_raw["original_split"].value_counts())

print("\nRows by subject:")
display(df_raw["subject"].value_counts().head(30))

print("\nAnswer distribution:")
display(df_raw["answer"].value_counts(dropna=False))

Rows by split:


original_split
test          13062
validation     1456
dev             271
Name: count, dtype: int64


Rows by subject:


subject
professional_law                       1247
miscellaneous                           868
moral_scenarios                         811
professional_psychology                 676
high_school_psychology                  601
high_school_macroeconomics              438
elementary_mathematics                  422
moral_disputes                          364
prehistory                              359
philosophy                              348
high_school_biology                     346
nutrition                               343
professional_accounting                 318
high_school_mathematics                 303
clinical_knowledge                      299
professional_medicine                   289
high_school_microeconomics              268
conceptual_physics                      266
marketing                               264
human_aging                             248
high_school_statistics                  244
high_school_chemistry                   229
high_school_geography   


Answer distribution:


answer
D    3943
C    3783
B    3660
A    3403
Name: count, dtype: int64

In [52]:
# tiền xử lý và chuẩn hoá
import numpy as np
import ast

ANSWER_LETTERS = ["A", "B", "C", "D"]

def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_answer(ans):
    """
    Robust cho cả dạng:
    - 'A'/'B'/'C'/'D'
    - 0/1/2/3
    - '0'/'1'/'2'/'3'
    """
    if ans is None:
        return None

    if isinstance(ans, int):
        if 0 <= ans <= 3:
            return ANSWER_LETTERS[ans]
        return None

    s = str(ans).strip().upper()

    if s in ANSWER_LETTERS:
        return s

    if s in ["0", "1", "2", "3"]:
        return ANSWER_LETTERS[int(s)]

    m = re.match(r"^([ABCD])[\.\)]?$", s)
    if m:
        return m.group(1)

    return None

def normalize_choices(choices):
    """
    Accept các dạng thường gặp:
    - list
    - tuple
    - numpy.ndarray
    - pandas Series
    - string biểu diễn list
    """
    if choices is None:
        return None

    if isinstance(choices, np.ndarray): # vì datasets -> pandas sẽ cho ra data kiểu ndarray
        choices = choices.tolist()

    elif isinstance(choices, pd.Series):
        choices = choices.tolist()

    elif isinstance(choices, tuple):
        choices = list(choices)

    elif isinstance(choices, str):
        s = choices.strip()
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                choices = list(parsed)
            else:
                return None
        except Exception:
            return None

    if not isinstance(choices, list):
        return None

    if len(choices) != 4:
        return None

    cleaned = [clean_text(c) for c in choices]

    if any(not c for c in cleaned):
        return None

    return cleaned

def make_user_content(question, choices):
    return "\n".join([
        f"Câu hỏi: {question}",
        f"A. {choices[0]}",
        f"B. {choices[1]}",
        f"C. {choices[2]}",
        f"D. {choices[3]}",
    ])

def make_hash(question, choices):
    key = json.dumps(
        {
            "question": question,
            "choices": choices,
        },
        ensure_ascii=False,
        sort_keys=True,
    )
    return hashlib.md5(key.encode("utf-8")).hexdigest()

In [53]:
def convert_row(row):
    question = clean_text(row.get("question"))
    choices = normalize_choices(row.get("choices"))
    answer = normalize_answer(row.get("answer"))

    reject_reasons = []

    if not question:
        reject_reasons.append("missing_question")

    if choices is None:
        reject_reasons.append("invalid_choices")

    if answer is None:
        reject_reasons.append("invalid_answer")

    if reject_reasons:
        return None, reject_reasons

    user_content = make_user_content(question, choices)

    item = {
        "messages": [
            {
                "role": "user",
                "content": user_content,
            },
            {
                "role": "assistant",
                "content": f"Đáp án: {answer}",
            },
        ],
        "metadata": {
            "task": TASK_NAME,
            "source": SOURCE_NAME,
            "difficulty": DIFFICULTY,
            "source_dataset": DATASET_ID,
            "source_config": CONFIG_NAME,
            "source_split": row.get("original_split"),
            "source_id": row.get("id"),
            "subject": row.get("subject"),
            "answer": answer,
        },
    }

    item["metadata"]["dedup_hash"] = make_hash(question, choices)
    return item, []

In [54]:
test_item, test_reasons = convert_row(df_raw.iloc[0].to_dict())

print("Reject reasons:", test_reasons)
print(json.dumps(test_item, ensure_ascii=False, indent=2))

Reject reasons: []
{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.\nA. Sai, Sai\nB. Sai, Không sai\nC. Không sai, Sai\nD. Không sai, Không sai"
    },
    {
      "role": "assistant",
      "content": "Đáp án: B"
    }
  ],
  "metadata": {
    "task": "wiki_mcq",
    "source": "lighteval/okapi_mmlu:vi",
    "difficulty": "medium",
    "source_dataset": "lighteval/okapi_mmlu",
    "source_config": "vi",
    "source_split": "test",
    "source_id": "moral_scenarios/test/106",
    "subject": "moral_scenarios",
    "answer": "B",
    "dedup_hash": "8202ae1a3d522b51c94e20dc8f4015a9"
  }
}


In [55]:
choice_type_counter = Counter(type(x).__name__ for x in df_raw["choices"])

print("Choice type distribution:")
display(pd.Series(choice_type_counter).sort_values(ascending=False))

Choice type distribution:


ndarray    14789
dtype: int64

In [56]:
# Convert toàn bộ + collect rejects
converted = []
rejects = []

for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
    item, reasons = convert_row(row.to_dict())

    if item is None:
        reject = row.to_dict()
        reject["_row_index"] = int(idx)
        reject["_reject_reasons"] = reasons
        rejects.append(reject)
    else:
        converted.append(item)

print("Rows before filter:", len(df_raw))
print("Rows after basic filter:", len(converted))
print("Rejected:", len(rejects))

  0%|          | 0/14789 [00:00<?, ?it/s]

Rows before filter: 14789
Rows after basic filter: 14789
Rejected: 0


In [57]:
seen = set()
deduped = []
duplicates = []

for item in converted:
    h = item["metadata"]["dedup_hash"]

    if h in seen:
        duplicates.append(item)
        continue

    seen.add(h)
    deduped.append(item)

print("Rows before dedup:", len(converted))
print("Rows after dedup:", len(deduped))
print("Duplicates removed:", len(duplicates))

Rows before dedup: 14789
Rows after dedup: 14787
Duplicates removed: 2


In [58]:
from collections import defaultdict

dup_groups = defaultdict(list)

for item in converted:
    h = item["metadata"]["dedup_hash"]
    dup_groups[h].append(item)

dup_groups = {
    h: items
    for h, items in dup_groups.items()
    if len(items) > 1
}

print("Number of duplicate groups:", len(dup_groups))
print("Total duplicate rows involved:", sum(len(items) for items in dup_groups.values()))
print("Rows to remove if keeping first:", sum(len(items) - 1 for items in dup_groups.values()))

Number of duplicate groups: 2
Total duplicate rows involved: 4
Rows to remove if keeping first: 2


In [59]:
dup_preview_rows = []

for h, items in dup_groups.items():
    for rank, item in enumerate(items):
        meta = item["metadata"]
        user_content = item["messages"][0]["content"]
        assistant_content = item["messages"][1]["content"]

        # lấy phần câu hỏi trước A.
        question_part = user_content.split("\nA.")[0].replace("Câu hỏi:", "").strip()

        dup_preview_rows.append({
            "dedup_hash": h,
            "dup_rank": rank,
            "source_split": meta.get("source_split"),
            "source_id": meta.get("source_id"),
            "subject": meta.get("subject"),
            "answer": meta.get("answer"),
            "assistant": assistant_content,
            "question": question_part[:300],
        })

df_dups = pd.DataFrame(dup_preview_rows)
display(df_dups)

,dedup_hash,dup_rank,source_split,source_id,subject,answer,assistant,question
0,682805ac6ce96f3e210f7096a917df66,0,test,clinical_knowledge/test/47,clinical_knowledge,A,Đáp án: A,Các loại steroid tổng hợp tăng cường hiệu suất...
1,682805ac6ce96f3e210f7096a917df66,1,test,college_medicine/test/9,college_medicine,A,Đáp án: A,Các loại steroid tổng hợp tăng cường hiệu suất...
2,cefc286c66c117447747782803ecde85,0,test,public_relations/test/3,public_relations,B,Đáp án: B,Năm nào BBC bắt đầu phát thanh đài phát thanh?
3,cefc286c66c117447747782803ecde85,1,test,public_relations/test/12,public_relations,B,Đáp án: B,Năm nào BBC bắt đầu phát thanh đài phát thanh?


In [60]:
def extract_answer_from_item(item):
    content = item["messages"][1]["content"]
    m = re.match(r"^Đáp án:\s*([ABCD])$", content)
    return m.group(1) if m else None

answer_counter = Counter()
subject_counter = Counter()
split_counter = Counter()

bad_assistant_format = 0
bad_user_format = 0

for item in deduped:
    ans = extract_answer_from_item(item)
    if ans is None:
        bad_assistant_format += 1
    else:
        answer_counter[ans] += 1

    subject_counter[item["metadata"].get("subject")] += 1
    split_counter[item["metadata"].get("source_split")] += 1

    user_content = item["messages"][0]["content"]
    for prefix in ["Câu hỏi:", "A.", "B.", "C.", "D."]:
        if prefix not in user_content:
            bad_user_format += 1
            break

print("Bad assistant format:", bad_assistant_format)
print("Bad user format:", bad_user_format)

print("\nAnswer distribution:")
display(pd.Series(answer_counter).sort_index())

print("\nSplit distribution:")
display(pd.Series(split_counter).sort_values(ascending=False))

print("\nTop subjects:")
display(pd.Series(subject_counter).sort_values(ascending=False).head(30))

Bad assistant format: 0
Bad user format: 0

Answer distribution:


A    3402
B    3659
C    3783
D    3943
dtype: int64


Split distribution:


test          13060
validation     1456
dev             271
dtype: int64


Top subjects:


professional_law                       1247
miscellaneous                           868
moral_scenarios                         811
professional_psychology                 676
high_school_psychology                  601
high_school_macroeconomics              438
elementary_mathematics                  422
moral_disputes                          364
prehistory                              359
philosophy                              348
high_school_biology                     346
nutrition                               343
professional_accounting                 318
high_school_mathematics                 303
clinical_knowledge                      299
professional_medicine                   289
high_school_microeconomics              268
conceptual_physics                      266
marketing                               264
human_aging                             248
high_school_statistics                  244
high_school_chemistry                   229
high_school_geography           

In [61]:
for i in range(3):
    print(json.dumps(deduped[i], ensure_ascii=False, indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.\nA. Sai, Sai\nB. Sai, Không sai\nC. Không sai, Sai\nD. Không sai, Không sai"
    },
    {
      "role": "assistant",
      "content": "Đáp án: B"
    }
  ],
  "metadata": {
    "task": "wiki_mcq",
    "source": "lighteval/okapi_mmlu:vi",
    "difficulty": "medium",
    "source_dataset": "lighteval/okapi_mmlu",
    "source_config": "vi",
    "source_split": "test",
    "source_id": "moral_scenarios/test/106",
    "subject": "moral_scenarios",
    "answer": "B",
    "dedup_hash": "8202ae1a3d522b51c94e20dc8f4015a9"
  }
}
{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Nhóm Quốc gia Islam kêu gọi đ

In [62]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(OUT_JSONL, deduped)
write_jsonl(OUT_REJECTS_JSONL, rejects)

print("Wrote:", OUT_JSONL)
print("Rows:", len(deduped))
print("Rejects:", OUT_REJECTS_JSONL, len(rejects))

Wrote: seed_exports/wiki_mcq_seed.jsonl
Rows: 14787
Rejects: seed_exports/wiki_mcq_seed_rejects.jsonl 0


In [63]:

top_subjects_series = pd.Series(subject_counter).sort_values(ascending=False).head(50)

report = {
    "dataset_id": DATASET_ID,
    "config": CONFIG_NAME,
    "task": TASK_NAME,
    "source": SOURCE_NAME,

    "rows_before_filter": int(len(df_raw)),
    "rows_after_basic_filter": int(len(converted)),
    "rows_after_dedup": int(len(deduped)),
    "rejected": int(len(rejects)),
    "duplicates_removed": int(len(duplicates)),

    "answer_distribution": {
        str(k): int(v)
        for k, v in sorted(answer_counter.items())
    },

    "split_distribution": {
        str(k): int(v)
        for k, v in split_counter.items()
    },

    "top_subjects": {
        str(k): int(v)
        for k, v in top_subjects_series.items()
    },

    "output_jsonl": str(OUT_JSONL),
    "rejects_jsonl": str(OUT_REJECTS_JSONL),
}

with open(OUT_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "dataset_id": "lighteval/okapi_mmlu",
  "config": "vi",
  "task": "wiki_mcq",
  "source": "lighteval/okapi_mmlu:vi",
  "rows_before_filter": 14789,
  "rows_after_basic_filter": 14789,
  "rows_after_dedup": 14787,
  "rejected": 0,
  "duplicates_removed": 2,
  "answer_distribution": {
    "A": 3402,
    "B": 3659,
    "C": 3783,
    "D": 3943
  },
  "split_distribution": {
    "test": 13060,
    "validation": 1456,
    "dev": 271
  },
  "top_subjects": {
    "professional_law": 1247,
    "miscellaneous": 868,
    "moral_scenarios": 811,
    "professional_psychology": 676,
    "high_school_psychology": 601,
    "high_school_macroeconomics": 438,
    "elementary_mathematics": 422,
    "moral_disputes": 364,
    "prehistory": 359,
    "philosophy": 348,
    "high_school_biology": 346,
    "nutrition": 343,
    "professional_accounting": 318,
    "high_school_mathematics": 303,
    "clinical_knowledge": 299,
    "professional_medicine": 289,
    "high_school_microeconomics": 268,
    "co

In [64]:
loaded_back = []

with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        loaded_back.append(json.loads(line))

print("Reloaded rows:", len(loaded_back))
print("First row:")
print(json.dumps(loaded_back[0], ensure_ascii=False, indent=2))

Reloaded rows: 14787
First row:
{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.\nA. Sai, Sai\nB. Sai, Không sai\nC. Không sai, Sai\nD. Không sai, Không sai"
    },
    {
      "role": "assistant",
      "content": "Đáp án: B"
    }
  ],
  "metadata": {
    "task": "wiki_mcq",
    "source": "lighteval/okapi_mmlu:vi",
    "difficulty": "medium",
    "source_dataset": "lighteval/okapi_mmlu",
    "source_config": "vi",
    "source_split": "test",
    "source_id": "moral_scenarios/test/106",
    "subject": "moral_scenarios",
    "answer": "B",
    "dedup_hash": "8202ae1a3d522b51c94e20dc8f4015a9"
  }
}


In [65]:
# Kiểm tra bắt buộc schema messages
schema_errors = []

for i, item in enumerate(loaded_back):
    if "messages" not in item or "metadata" not in item:
        schema_errors.append((i, "missing_top_level_keys"))
        continue

    messages = item["messages"]
    if not isinstance(messages, list) or len(messages) != 2:
        schema_errors.append((i, "invalid_messages"))
        continue

    if messages[0].get("role") != "user":
        schema_errors.append((i, "invalid_user_role"))

    if messages[1].get("role") != "assistant":
        schema_errors.append((i, "invalid_assistant_role"))

    if not re.match(r"^Đáp án:\s*[ABCD]$", messages[1].get("content", "")):
        schema_errors.append((i, "invalid_assistant_content"))

print("Schema errors:", len(schema_errors))
schema_errors[:10]

Schema errors: 0


[]